# Edge vision adaptation test: a vision-language model plus a small classifier, on a mock workstation

**By** Chander Maurya Kumbkarni · September 2026 · the case study in this repository's README covers the design and the results.

This is the experiment behind the case study. A phone on a stand filmed a mock workstation in 13 clips: a steel tray as the fixture, a padlock as part A (the old part) and a whisk as part B (the new part). Every approach answers one question on every frame: **is there a part on the tray?** It asks about the fixture, not the product, so a new part does not change the question.

| | Approach | How it adapts to a change |
|---|---|---|
| 1 | **YOLO classifier** (`yolo11n-cls`), a stand-in for a fine-tuned detector and its retrain workflow | retrain on labelled frames |
| 2 | **Fast classifier:** a frozen image encoder (SigLIP 2) plus a logistic regression | refit in milliseconds on stored image embeddings |
| 3 | **Checker A:** Qwen3.5-0.8B reading its probability of "Yes" in one forward pass; it writes nothing | none, never trained |
| 4 | **Checker B:** Intern-Decision-0.8B (InternLM, released 26 Sep 2026), a vision model built to answer fixed questions with probabilities | none, never trained |

**Four changes:** a new product, the lighting (both ways), a moved camera and a new station. For each: the accuracy right after the change with nothing retrained, and the labels and seconds needed to get back to 95%. Then two questions about the checkers (can they tell loaded from empty at all, and does a per-site cutoff fix them), and the speed budget for 10 cameras at 0.5 FPS on a Jetson AGX Orin 32GB.

**Run it:** Runtime > Change runtime type > **T4 GPU** > Save, then Runtime > **Run all**. The 13 clips (about 285 MB) download from this repository's release by themselves. The run takes about 30 minutes, mostly model downloads and YOLO retrains, and the last cell downloads a zip of the results.

**Limits:** one camera, one scene, one afternoon. A YOLO classifier stands in for a production detector. The accuracy numbers will not transfer to a real machine cell; the method, the checks and the speed ratios should. Numbers can move by a frame or two between runs, because YOLO training on a GPU is not bit-for-bit repeatable.

In [ ]:
DEMO = False                              # False: the 13 phone clips from this repository's release. True: 13 synthetic clips, only to test the pipeline
CLIPS_URL = "https://github.com/akamaurya/edge-vision-adaptation-test/releases/download/clips-v1/"   # the 13 clips are attached to a release of this repository
CLIP_NAMES = [f"{g}_{l}.MOV" for g in ("day", "night", "moved", "s2") for l in ("empty", "A", "B")] + ["day_cycle.MOV"]
VLM_ID = "Qwen/Qwen3.5-0.8B"              # checker A. A bigger option for a second run: "Qwen/Qwen3.5-2B"
IDM_ID = "internlm/Intern-Decision-0.8B"  # checker B, optional: the notebook runs without it
ENC_ID = "google/siglip2-base-patch16-224"
QUESTION = "This photo shows a metal tray on a table, seen from a fixed camera. Is the tray empty? Answer Yes or No."   # checkers: P(part) = 1 - P(yes)
FPS = 2                                   # frames pulled per second of video
SKIP_S = 10                               # seconds dropped at the start of every clip: the part was still being placed
YOLO_EPOCHS = 15
NS, YOLO_NS, SEEDS = (0, 5, 10, 20, 50), (20, 50), range(5)   # labels added in the recovery curves; seeds for the fast classifier
YOLO_NS_NEW_PART = (5, 10, 20)            # extra YOLO retrains for the new part, so YOLO and the fast classifier meet at the same label counts
CUTOFF_N = 20                             # labelled frames used to fit a per-site cutoff for each checker (section 11)
TARGET = 0.95                             # "recovered" = at least 95% on that condition's test frames
UNSURE = (0.2, 0.8)                       # fast-classifier P(part) in this band = unsure, goes to the checker
FAST_CS = (1, 3, 10, 30, 100)             # candidate strengths for the fast classifier; one is picked on training frames only (section 4)
AUDIT = 0.03                              # share of the other frames the checker re-checks at random
ORIN_FACTOR = 2.5                         # assumed slowdown, T4 to AGX Orin 32GB (see the README)
CAMERAS, CAMERA_FPS = 10, 0.5             # the brief: 10 cameras at 0.5 FPS

In [ ]:
!pip -q install -U transformers accelerate ultralytics

## 1. Clips to frames
The 13 clips download from this repository's release (`CLIPS_URL` above). If a download fails, the cell asks you to upload them instead. Clip names carry the labels:

| Clips | Station | Light |
|---|---|---|
| `night_empty`, `night_A`, `night_B` | station 1 | lamp |
| `day_empty`, `day_A`, `day_B` | station 1 | daylight |
| `day_cycle` | station 1, part B loaded and unloaded twice | daylight |
| `moved_empty`, `moved_A`, `moved_B` | station 1 with the camera nudged | lamp |
| `s2_empty`, `s2_A`, `s2_B` | station 2: a different table, tray and background | lamp |

Frames are pulled at 2 per second. The first 10 seconds of every clip are dropped: in the part clips the part was still being placed, so those frames don't match the clip's label. An earlier run kept them, and that label noise made YOLO look broken on the new part. The rest of each clip is split by time: the first 70% can be used for training, the last 30% is test. **All of `day_B` is test:** nobody trains on the new part in daylight.

In [ ]:
import os, sys, glob, json, shutil, subprocess, time
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from PIL import Image

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all"
RUN = "demo" if DEMO else "real"
CLIPS, FRAMES, OUT = f"clips_{RUN}", f"frames_{RUN}", f"results_{RUN}"
for d in (CLIPS, FRAMES, OUT): os.makedirs(d, exist_ok=True)
GROUPS = {"day": ("s1", "day"), "night": ("s1", "night"), "moved": ("s1_moved", "night"), "s2": ("s2", "night")}   # clip prefix: station, light
VIDEO = (".mov", ".mp4", ".m4v", ".avi", ".mkv")
clip_files = lambda: sorted(p for p in glob.glob(f"{CLIPS}/**/*", recursive=True) if os.path.splitext(p)[1].lower() in VIDEO)

def make_demo_clips(folder, fps=5):
    # synthetic stand-ins for all 13 clips: station 1 by day and lamp light, the same scene with the camera
    # shifted and turned, and a second station in other colours
    rng = np.random.default_rng(0)
    clips = [(g, lab, 20) for g in GROUPS for lab in ("empty", "A", "B")] + [("day", "cycle", 24)]
    for group, label, secs in clips:
        vw = cv2.VideoWriter(f"{folder}/{group}_{label}.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (640, 360))
        for t in range(secs * fps):
            part = "B" if label == "cycle" and (4 <= t / fps < 10 or 14 <= t / fps < 20) else label   # cycle: 2 loads
            s2 = group == "s2"
            img = np.full((360, 640, 3), (60, 95, 70) if s2 else (90, 110, 130), np.uint8)          # table
            if s2: img[:70] = (40, 40, 60)                                                          # a shelf behind station 2
            tray = ((200, 100), (440, 300), (120, 170, 210)) if s2 else ((220, 110), (420, 290), (200, 200, 200))
            cv2.rectangle(img, tray[0], tray[1], tray[2], -1)
            dx, dy = (int(v) for v in rng.integers(-15, 16, 2))
            if part == "A": cv2.circle(img, (320 + dx, 200 + dy), 45, (140, 60, 30), -1)
            if part == "B": cv2.rectangle(img, (285 + dx, 170 + dy), (365 + dx, 230 + dy), (40, 40, 200), -1)
            if group == "day":
                img = cv2.convertScaleAbs(img, alpha=1.3, beta=40)
                cv2.circle(img, ((250 + 3 * t) % 640, 90), 50, (255, 255, 255), -1)    # moving glare
            else:
                img = cv2.convertScaleAbs(img * np.array([0.55, 0.7, 0.9]))           # dim, warm lamp
            if group == "moved":                                                     # camera bumped: turned 4 degrees and shifted
                M = cv2.getRotationMatrix2D((320, 180), 4, 1.0); M[:, 2] += (25, 12)
                img = cv2.warpAffine(img, M, (640, 360), borderMode=cv2.BORDER_REPLICATE)
            vw.write(cv2.add(img, rng.integers(0, 12, img.shape, dtype=np.uint8)))
        vw.release()

if DEMO:
    make_demo_clips(CLIPS)
elif not clip_files():
    import urllib.request
    for name in CLIP_NAMES:
        try:
            urllib.request.urlretrieve(CLIPS_URL + name, f"{CLIPS}/{name}")
        except Exception as ex:
            print(f"could not download {name}: {str(ex)[:150]}")
    for p in clip_files():
        if os.path.getsize(p) < 1_000_000: os.remove(p)                      # an error page, not a video
    print(f"Downloaded {len(clip_files())} of {len(CLIP_NAMES)} clips")
    if len(clip_files()) < len(CLIP_NAMES):
        print("Some clips are missing. Upload them now, or download them from the repository's release page and drag them into", CLIPS, "then Run all again")
        from google.colab import files
        for name, data in files.upload().items():
            open(f"{CLIPS}/{name}", "wb").write(data)

rows, cycle = [], []
for path in clip_files():
    group, label = (os.path.splitext(os.path.basename(path))[0].lower().split("_") + [""])[:2]
    label = {"empty": "empty", "a": "A", "b": "B", "cycle": "cycle"}.get(label)
    if group not in GROUPS or not label or (label == "cycle" and group != "day"):
        print("skipping (name it like day_A.mov or s2_B.mov):", path); continue
    clip = f"{group}_{label}"
    out = f"{FRAMES}/{clip}"
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", path, "-vf", f"fps={FPS},scale=640:-2", f"{out}/%04d.jpg"], check=True)
    fs = sorted(glob.glob(f"{out}/*.jpg"))
    if label == "cycle":
        cycle = fs; continue
    fs = fs[int(SKIP_S * FPS):]                                                  # the label is per clip, so skip the seconds before the part was on the tray
    station, light = GROUPS[group]
    rows += [dict(path=f, clip=clip, station=station, light=light, label=label, pool="train" if i < 0.7 * len(fs) else "test") for i, f in enumerate(fs)]

df = pd.DataFrame(rows)
df["part"] = df.label != "empty"
df.loc[df["clip"] == "day_B", "pool"] = "test"                                     # the failure case: never trained on
print(df.groupby(["station", "light", "label", "pool"]).size().unstack(fill_value=0))
assert len(df[df.station == "s1"].groupby(["light", "label"])) == 6, "Need the 6 base clips: day/night x empty/A/B"
HAVE = {st: df[df.station == st].label.nunique() == 3 for st in ("s1_moved", "s2")}   # an optional group needs empty, A and B
for st, ok in HAVE.items():
    if not ok: print(f"Note: {st} clips missing or incomplete, so that test is skipped and stored as null.")
if not cycle: print("Note: no day_cycle clip, so the cycle plot is skipped.")


## 2. The balance check, before any training
Before any training: count frames per part and lighting, and flag any cell that is missing or under half the biggest one.
A training set where the new part has far fewer frames than the old one, or is missing in one light, is a common way to lose a retrain. This check flags that dataset automatically, before the retrain starts.
**Here it only reports.** We train anyway, on purpose, to see what the lopsided set does. In a deployment it becomes a hard stop before any expensive retrain.

In [ ]:
def balance_check(train, name):
    t = train.groupby(["label", "light"]).size().unstack(fill_value=0)
    t = t.reindex(columns=["day", "night"], fill_value=0)
    gaps = [f"{lab} has {t.loc[lab, li]} {li} frames (largest cell {t.values.max()})"
            for lab in t.index for li in t.columns if t.loc[lab, li] < 0.5 * t.values.max()]
    print(f"{name}: BALANCE CHECK {'PASS' if not gaps else 'FAIL'}"); print(t)
    for g in gaps: print("  -", g)
    return not gaps

s1_train = df.query("station == 's1' and pool == 'train'")
yolo_train = s1_train                                                # includes B under lamp light only
head_train = s1_train[s1_train.label != "B"]                         # no part B at all
balance_check(yolo_train, "YOLO training set (new part filmed in one light only)")
print("  -> In a deployment this stops the retrain until part B is filmed in daylight. Here we train anyway, to see what happens.")
balance_check(head_train, "Fast-classifier training set (it answers part / no part, so it doesn't need part B)")

## 3. Approach 1: a YOLO classifier, standing in for a fine-tuned detector
Same model family and the same retraining workflow as a production part detector, but a classifier (classes: empty, A, B), not a detector. This first model is trained on station 1 with part B under lamp light only. Every later "YOLO retrain" repeats this training on other frames and records the seconds.

In [ ]:
from ultralytics import YOLO

def yolo_folder(train, root=f"yolo_{RUN}"):
    shutil.rmtree(root, ignore_errors=True)
    for _, g in train.groupby("label"):
        for i, r in enumerate(g.itertuples()):
            for split in (("train", "val") if i % 5 == 0 else ("train",)):    # every 5th frame of each class also goes to a val folder, which Ultralytics needs for its logs
                d = f"{root}/{split}/{r.label}"; os.makedirs(d, exist_ok=True)
                shutil.copy(r.path, f"{d}/{r.clip}_{os.path.basename(r.path)}")
    return os.path.abspath(root)

def train_yolo(frames):                                              # fine-tune yolo11n-cls on these frames: the model and the seconds it took
    t0 = time.time()
    m = YOLO("yolo11n-cls.pt")
    m.train(data=yolo_folder(frames), epochs=YOLO_EPOCHS, imgsz=224, batch=32, plots=False, verbose=False)
    return YOLO(m.trainer.best), time.time() - t0

def yolo_part(model, frames):                                        # True = YOLO's top answer is part A or part B
    return np.array([r.names[r.probs.top1] != "empty" for r in model.predict(list(frames.path), imgsz=224, verbose=False, stream=True)])

def yolo_p_part(model, frames):                                      # YOLO's probability of "part" = 1 - P(empty), for plots and the video
    res = list(model.predict(list(frames.path), imgsz=224, verbose=False, stream=True))
    e = next(k for k, v in res[0].names.items() if v == "empty")
    return np.array([1 - float(r.probs.data[e]) for r in res])

yolo, yolo_train_s = train_yolo(yolo_train)
df["yolo"] = yolo_part(yolo, df)
df["yolo_p"] = yolo_p_part(yolo, df)
print(f"YOLO retrain took {yolo_train_s:.0f} s on this GPU")

## 4. Approach 2: the fast classifier
A frozen image encoder (SigLIP 2, the same family as the eyes inside many VLMs) turns each frame into 768 numbers, once. Only a logistic regression on top learns *part vs no part*. Every refit below reuses the stored numbers, which is why a refit takes milliseconds. Its one setting, how strongly the logistic regression is regularised, is picked on the later training frames of each station 1 clip, never on test frames.
Two station 1 models: one that never saw part B (for the new-product test), and one trained on the same frames as YOLO (the starting point for the camera-moved and new-station tests).

In [ ]:
from transformers import AutoModel, AutoProcessor
from sklearn.linear_model import LogisticRegression

enc = AutoModel.from_pretrained(ENC_ID, dtype=torch.float16).cuda().eval()
enc_proc = AutoProcessor.from_pretrained(ENC_ID)

@torch.no_grad()
def embed(paths, bs=32):
    out = []
    for i in range(0, len(paths), bs):
        x = enc_proc(images=[Image.open(p).convert("RGB") for p in paths[i:i + bs]], return_tensors="pt").to("cuda", torch.float16)
        e = enc.get_image_features(**x)
        e = getattr(e, "pooler_output", e)                                     # newer transformers return an output object
        out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

In [ ]:
t0 = time.time()
E = embed(list(df.path))                                             # every frame embedded once; every refit below reuses these
embed_ms = (time.time() - t0) / len(df) * 1000
assert np.isfinite(E).all(), "SigLIP 2 embeddings are not finite (fp16 overflow?)"

# pick the regularisation on station 1's training frames only: fit on the earlier frames of each clip, score on the later ones
from sklearn.metrics import log_loss
k, n = yolo_train.groupby("clip").cumcount(), yolo_train.groupby("clip")["clip"].transform("size")
early, late = yolo_train[k < n * 5 / 7], yolo_train[k >= n * 5 / 7]
val_loss = {C: log_loss(late.part, LogisticRegression(C=C, max_iter=5000).fit(E[early.index], early.part).predict_proba(E[late.index])[:, 1], labels=[False, True]) for C in FAST_CS}
FAST_C = min(val_loss, key=val_loss.get)
print(f"Fast-classifier C = {FAST_C}, picked on training frames only. Validation log loss per C: { {C: round(v, 3) for C, v in val_loss.items()} }")

def fit(frames, y="part"):                                           # the fast classifier: logistic regression on stored embeddings
    return LogisticRegression(C=FAST_C, max_iter=5000).fit(E[frames.index], frames[y])   # one strength per run here; one per station in production

def p_part(model, frames):
    return model.predict_proba(E[frames.index])[:, 1]

t0 = time.time()
head = fit(head_train)                                               # station 1, never saw part B
head_fit_s = time.time() - t0
head_s1 = fit(yolo_train)                                            # station 1, same frames as YOLO
df["head_p"], df["head_s1_p"] = p_part(head, df), p_part(head_s1, df)
df["head"] = df.head_p > 0.5
print(f"Embedding: {embed_ms:.1f} ms per frame, done once for {len(df)} frames. Fast-classifier fit on {len(head_train)} frames: {head_fit_s:.2f} s")

## 5. Approach 3: checker A, a small VLM that decides instead of talking
One forward pass per frame, then read the model's probability of **Yes** vs **No** for the next token. Nothing is generated, so it can't ramble or break a format. It runs in fp16 (the T4 has no bf16), and the logits are checked to be finite.
Both checkers are asked "is the tray empty?" (on a real machine: "is the fixture empty?"), so P(part) = 1 - P(yes). On a sample of training frames this phrasing beat "is there an object on the tray?": the steel tray itself, with its glare, reads as "an object".
The checkers answer every frame (about 520), so section 11 can fit a per-site cutoff on training frames and test it on test frames.

In [ ]:
import transformers

def load_vlm(dtype):
    for cls in ("AutoModelForImageTextToText", "AutoModelForMultimodalLM"):     # name differs across transformers versions
        try:
            return getattr(transformers, cls).from_pretrained(VLM_ID, dtype=dtype, device_map="cuda").eval()
        except Exception as ex:
            print(cls, "did not load:", str(ex)[:200])
    raise RuntimeError(f"could not load {VLM_ID}")

vlm = load_vlm(torch.float16)
vproc = AutoProcessor.from_pretrained(VLM_ID)
vproc.tokenizer.padding_side = "left"
tok = vproc.tokenizer
YES = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ("Yes", "yes")})
NO = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ("No", "no")})

def prompt(q):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]}]
    return vproc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

def vlm_inputs(paths, q):
    ims = [Image.open(p).convert("RGB") for p in paths]
    return vproc(text=[prompt(q)] * len(ims), images=ims, return_tensors="pt", padding=True).to("cuda")

@torch.no_grad()
def last_logits(x):
    try: return vlm(**x, logits_to_keep=1).logits[:, -1].float()
    except TypeError: return vlm(**x).logits[:, -1].float()

def p_yes(paths, q=QUESTION, bs=10):
    out = []
    for i in range(0, len(paths), bs):
        lg = last_logits(vlm_inputs(paths[i:i + bs], q))
        out += torch.sigmoid(torch.logsumexp(lg[:, YES], -1) - torch.logsumexp(lg[:, NO], -1)).tolist()
    return np.array(out)

x = vlm_inputs([df.path.iloc[0]], QUESTION)
lg = last_logits(x)
if not torch.isfinite(lg).all():                                     # fp16 can overflow; the T4 has no bf16, so fall back to float32
    print("Checker A: fp16 logits not finite, reloading in float32 (slower, same model)")
    del vlm; torch.cuda.empty_cache(); vlm = load_vlm(torch.float32); lg = last_logits(x)
assert torch.isfinite(lg).all(), "Checker A logits are not finite"
TOKENS_PER_FRAME = int(x["input_ids"].shape[1])
print("tokens per frame (image + question):", TOKENS_PER_FRAME)
print("top next tokens:", [tok.decode(int(t)) for t in lg[0].topk(5).indices])

In [ ]:
def p_part_A(paths):                                                 # the question is "is the tray empty?", so P(part) = 1 - P(yes)
    return 1 - p_yes(paths)

t0 = time.time()
df["vlm_p"] = p_part_A(list(df.path))                                # every frame, so section 11 can fit a cutoff on training frames
vlm_ms = (time.time() - t0) / len(df) * 1000
assert np.isfinite(df.vlm_p).all(), "Checker A gave a non-finite P(part)"
df["vlm"] = df.vlm_p > 0.5
print(f"Checker A answered {len(df)} frames, {vlm_ms:.0f} ms per frame")

## 5b. Approach 4: checker B, Intern-Decision-0.8B (optional)
A vision decision model built on Qwen3.5-0.8B: it answers 1 to 16 yes/no or multiple-choice questions about an image in one forward pass, with calibrated probabilities. Its loader ships inside the model repo. We don't install the repo's `requirements.txt`, because it pins exact torch and transformers versions that would replace Colab's.
It was released on 26 Sep 2026 and nobody has tested it on frames like these, so it is fully wrapped: if it fails to load or answers in an unexpected format, the notebook prints why and carries on without it. The first answer is printed in full so its format can be checked. Its loader takes one frame at a time, so it answers every frame one by one.

In [ ]:
STATE = "A fixed factory camera looks down at a workstation: a flat metal tray on a table. Parts are placed on the tray; an operator's hand may pass over it."
Q1 = {"empty": {"type": "noul", "instructions": "Is the tray empty?"}}   # P(part) = 1 - P(yes)
Q6 = {**Q1, **{k: {"type": "noul", "instructions": q} for k, q in {
    "hand": "Is a hand visible?", "tray": "Is the whole tray visible?", "daylight": "Is the scene lit by daylight?",
    "metal": "Is there a metal object on the tray?", "many": "Are there two or more objects on the tray?"}.items()}}

def idm_p(ans):                                                      # P(yes) from one answer; the model card puts it in "noul"
    p, probs = None, ans.get("probabilities") if isinstance(ans, dict) else None
    probs = {str(k).lower(): v for k, v in probs.items()} if isinstance(probs, dict) else {}
    if isinstance(ans, dict) and isinstance(ans.get("noul"), (int, float)): p = ans["noul"]
    elif "yes" in probs: p = probs["yes"]
    elif isinstance(ans, dict) and "decision" in ans and "confidence" in ans:
        p = ans["confidence"] if str(ans["decision"]).lower() in ("yes", "true") else 1 - ans["confidence"]
    elif isinstance(ans, (int, float)): p = ans
    if p is None or not np.isfinite(p) or not 0 <= p <= 1:
        raise ValueError(f"unrecognised Intern-Decision answer: {str(ans)[:300]}")
    return float(p)

def idm_p_part(paths, qs=Q1):
    return np.array([1 - idm_p(ask(p, qs)["answers"]["empty"]) for p in paths])

def load_engine(dtype):                                              # load, then check one real answer parses before trusting it
    e = DecisionEngine(checkpoint=IDM_DIR, device="cuda", dtype=dtype)   # explicit: its default path breaks on the HF cache's symlinks
    r = e.predict({"state": STATE, "questions": Q6, "images": [os.path.abspath(df.path.iloc[0])]})
    print(f"Checker B raw answer ({dtype}):", json.dumps(r, default=str)[:700])
    [idm_p(r["answers"][k]) for k in Q6]
    return e

CHECKER_B = False
try:
    from huggingface_hub import snapshot_download
    IDM_DIR = snapshot_download(IDM_ID)
    sys.path.insert(0, IDM_DIR)
    from inference import DecisionEngine                             # ships inside the model repo
    try:
        engine = load_engine("float16")                              # the T4 has no bf16
    except (ValueError, ArithmeticError) as ex:                      # e.g. non-finite probabilities in fp16
        print(f"Checker B in fp16 failed ({str(ex)[:200]}), retrying in float32")
        torch.cuda.empty_cache(); engine = load_engine("float32")
    ask = lambda path, qs: engine.predict({"state": STATE, "questions": qs, "images": [os.path.abspath(path)]})
    t0 = time.time()
    df["idm_p"] = idm_p_part(list(df.path))
    idm_ms = (time.time() - t0) / len(df) * 1000
    df["idm"] = df.idm_p > 0.5
    CHECKER_B = True
    print(f"Checker B answered {len(df)} frames, {idm_ms:.0f} ms per frame")
except Exception as ex:
    print(f"Checker B skipped ({type(ex).__name__}: {str(ex)[:300]}). Everything else runs without it.")

## 6. How each change is scored
For each change, two numbers per approach:
1. **Accuracy right after the change**, with nothing retrained.
2. **Recovery:** the fewest labels, and the seconds of compute, that bring accuracy back to at least 95% on that condition's test frames.

The recovery curves add N labelled frames, N in {0, 5, 10, 20, 50}, capped at what the clips provide. Frames are drawn evenly across classes with a fixed seed. The fast classifier repeats this over 5 seeds (mean and worst seed), because small N is noisy. YOLO retrains at N = 20 and 50 with one seed, because training is the slow part; for the new part it also retrains at 5, 10 and 20, so both approaches meet at the same label counts. Seconds are the classifier fit time or the YOLO training time. The checkers are never trained, so they get one number each.

Every result also shows **a model that always says "part"**: the score of guessing. A checker that can't beat it isn't seeing anything.

In [ ]:
curves, baselines, CHECKERS = {}, {}, [("vlm", "A")] + [("idm", "B")] * CHECKER_B

def pt(n, mean, s=0.0, lo=None):                                     # one point of a recovery curve
    return dict(n=int(n), mean=round(float(mean), 4), min=round(float(mean if lo is None else lo), 4), s=round(float(s), 3))

def acc(pred, frames):
    return float((np.asarray(pred) == frames.part.values).mean())

def always_part(frames):                                             # the score of a model that always says "part": what guessing gets
    return float(frames.part.mean())

def sample(pool, n, seed, by="label"):                               # n frames, spread evenly over the classes in `by` as far as the pool allows
    shuffled = pool.sample(frac=1, random_state=seed)
    return shuffled.iloc[np.argsort(shuffled.groupby(by).cumcount().values, kind="stable")[:n]]

def plus(base, extra):
    return pd.concat([base, extra]) if len(base) else extra

def Ns(pool, ns=NS):                                                 # N values above 0, capped at what the pool has
    return sorted({min(n, len(pool)) for n in ns} - {0})

def fc_curve(start, base, pool, test, y="part", by="label"):
    # fast classifier: N = 0 is `start` as it is; N > 0 refits on `base` plus N frames of `pool` (labels in column y), over 5 seeds
    out = [pt(0, acc(p_part(start, test) > 0.5, test))]
    for n in Ns(pool):
        a, s = [], []
        for seed in SEEDS:
            tr = plus(base, sample(pool, n, seed, by))
            t0 = time.time(); m = fit(tr, y); s.append(time.time() - t0)
            a.append(acc(p_part(m, test) > 0.5, test))
        out.append(pt(n, np.mean(a), np.mean(s), min(a)))
    return out

def yolo_curve(start, base, pool, test, ns=YOLO_NS):
    # YOLO: N = 0 is `start` as it is; each N retrains on `base` plus the seed-0 sample the fast classifier also gets
    out = [pt(0, acc(yolo_part(start, test), test))]
    for n in Ns(pool, ns):
        m, s = train_yolo(plus(base, sample(pool, n, 0)))
        out.append(pt(n, acc(yolo_part(m, test), test), s))
    return out

def teacher_curves(start, base, pool, test):
    # the fast classifier refit on N confident checker labels (P above 0.9 or below 0.1) instead of a person's
    out = {}
    for c, name in [("vlm", "A"), ("idm", "B")]:
        key = f"Fast classifier, checker {name} labels"
        if (c, name) not in CHECKERS:
            out[key] = None; continue
        conf = pool[(pool[f"{c}_p"] > 0.9) | (pool[f"{c}_p"] < 0.1)].assign(teach=lambda f: f[f"{c}_p"] > 0.5)
        if conf.teach.nunique() < 2:
            print(f"{key}: skipped, {len(conf)} confident labels and not both answers among them"); out[key] = None; continue
        out[key] = fc_curve(start, base.assign(teach=base.part), conf, test, y="teach", by="teach")
    return out

def checkers(frames):                                                # the checkers are never trained: one point, N = 0
    return {f"Checker {name}": [pt(0, acc(frames[c], frames))] if (c, name) in CHECKERS else None for c, name in [("vlm", "A"), ("idm", "B")]}

## 7. Test 1: new product
Part B arrives. The fast classifier never saw it. The YOLO stand-in was retrained with B under lamp light only. First the scoreboard: accuracy on *is there a part on the tray?* per lighting and part, same held-out station 1 frames for everyone.
Ship gate: a new approach ships only if no cell is more than 2 points worse than the current model and the average is at least as good.
Then the recovery, on the frames after a changeover (tray empty or holding part B). YOLO's "right after" is a model trained without B. Both YOLO and the fast classifier then get 5, 10, 20 and all 38 lamp-light frames of part B.

In [ ]:
M = ["yolo", "head", "vlm"] + ["idm"] * CHECKER_B
NAMES = {"yolo": "YOLO (retrained on B-lamp)", "head": "Fast classifier (no B)", "vlm": "Checker A (no training)", "idm": "Checker B (no training)",
         "always_part": "Always 'part' (guessing)"}
s1_test = df.query("station == 's1' and pool == 'test'")
score = s1_test.assign(**{m: s1_test[m] == s1_test.part for m in M}, always_part=s1_test.part).groupby(["light", "label"])[M + ["always_part"]].mean()
score.loc[("ALL", "mean"), :] = score.mean()
print((score * 100).round(1).rename(columns=NAMES).to_string())
print("\nFailure case, part B in daylight: " + " | ".join(f"{NAMES[m]} {score.loc[('day', 'B'), m]:.0%}" for m in M))

def ship_gate(new, old="yolo"):
    cells = score.drop(("ALL", "mean"))
    worse = cells[cells[new] < cells[old] - 0.02].index.tolist()
    ok = bool(not worse and cells[new].mean() >= cells[old].mean())
    print(f"SHIP GATE {new} vs {old}: {'SHIP' if ok else 'DO NOT SHIP'}" + (f"  (worse on {worse})" if worse else ""))
    return ok

gate = {m: ship_gate(m) for m in M[1:]}

NP = s1_test[s1_test.label != "A"]                                   # after the changeover the tray is empty or holds the new part
B_lamp = df.query("clip == 'night_B' and pool == 'train'")          # the only part B frames anyone may train on
yolo_noB, _ = train_yolo(head_train)                                 # YOLO before the retrain: never saw part B
yolo_np = yolo_curve(yolo_noB, head_train, B_lamp, NP, ns=YOLO_NS_NEW_PART)   # retrains on 5, 10 and 20 frames of part B
if all(p["n"] != len(B_lamp) for p in yolo_np):
    yolo_np.append(pt(len(B_lamp), acc(NP.yolo, NP), yolo_train_s))          # all of them: the model trained in section 3
curves["new product"] = {
    "YOLO retrain": yolo_np,
    "Fast classifier, human labels": fc_curve(head, head_train, B_lamp, NP),
    **checkers(NP)}
baselines["new product"] = always_part(NP)
print(f"\nNew product, accuracy right after the change (guessing 'part' scores {baselines['new product']:.0%}): "
      + ", ".join(f"{k} {v[0]['mean']:.0%}" for k, v in curves["new product"].items() if v))
for k in ("YOLO retrain", "Fast classifier, human labels"):
    print(f"  {k}: " + ", ".join(f"N={p['n']}: {p['mean']:.0%} ({p['s']:.2g} s)" for p in curves["new product"][k]))

## 7b. A load/unload cycle, new part, daylight
A monitoring product turns per-frame states into events (load, unload, cycle time). Here each approach's P(part) over time, smoothed over 3 frames (1.5 s) like an event stitcher would, and the loads it counts, with the time of each. The true answer is 2 loads. **A count can be right by luck,** so check the times against the plot.

In [ ]:
loads, load_times = None, None
if cycle:
    tl = pd.DataFrame({"frame": [os.path.basename(p) for p in cycle], "t_s": np.arange(len(cycle)) / FPS,
                       "YOLO": yolo_p_part(yolo, pd.DataFrame({"path": cycle})),
                       "Fast classifier": head.predict_proba(embed(cycle))[:, 1],
                       "Checker A": p_part_A(cycle)})
    if CHECKER_B: tl["Checker B"] = idm_p_part(cycle)
    models = [c for c in tl.columns if c not in ("frame", "t_s")]
    on = {m: (tl[m].rolling(3, center=True, min_periods=1).median() > 0.5).astype(int) for m in models}   # smoothed over 1.5 s, like an event stitcher
    load_times = {m: [float(t) for t in tl.t_s[on[m].diff() == 1]] for m in models}
    loads = {m: len(v) for m, v in load_times.items()}
    for m in models: print(f"{m}: {loads[m]} loads, starting at {load_times[m]} s")
    ax = tl.plot(x="t_s", y=models, figsize=(12, 3), ylim=(-0.05, 1.05), title=f"day_cycle, P(part on tray). Loads counted: {loads}. True: 2")
    ax.axhline(0.5, color="grey", lw=0.8, ls="--"); ax.set_xlabel("seconds")
    plt.tight_layout(); plt.savefig(f"{OUT}/cycle.png", dpi=150); plt.show()
    tl.to_csv(f"{OUT}/cycle.csv", index=False)
else:
    print("No day_cycle clip, skipping.")

## 8. Test 2: lighting
Train on lamp light only and test in daylight, then the reverse. Only the light changes: the daylight training frames have no part B (all of `day_B` is kept for testing), so the reverse direction leaves part B out of its test frames too. Recovery adds N labelled frames of the new light. The checkers are zero-shot.

In [ ]:
night_tr, day_tr = s1_train[s1_train.light == "night"], s1_train[s1_train.light == "day"]
day_te = s1_test[s1_test.light == "day"]
night_te = s1_test[(s1_test.light == "night") & (s1_test.label != "B")]   # daylight training has no B, so B stays out: only the light changes
for test, tr, te, pool in [("lighting, night to day", night_tr, day_te, day_tr),
                           ("lighting, day to night", day_tr, night_te, night_tr[night_tr.label != "B"])]:
    y0, _ = train_yolo(tr)
    curves[test] = {"YOLO retrain": yolo_curve(y0, tr, pool, te), "Fast classifier, human labels": fc_curve(fit(tr), tr, pool, te), **checkers(te)}
    baselines[test] = always_part(te)
    print(f"{test}, accuracy right after the change (guessing 'part' scores {baselines[test]:.0%}): "
          + ", ".join(f"{k} {v[0]['mean']:.0%}" for k, v in curves[test].items() if v))

## 9. Test 3: camera moved
The phone was nudged or tilted; tray, parts and lamp are unchanged. The station 1 models are applied to the moved frames as they are. Two things to watch:
- **Accuracy drop:** station 1 lamp-light test frames vs moved test frames.
- **Drift alarm:** the share of frames where the fast classifier is unsure (P(part) between 0.2 and 0.8). It fires when that share at least doubles and rises by 5 points or more. On site, this is how a bumped camera shows up in the morning report before a client notices.

Recovery adds N labelled moved frames to station 1's labels, from a person or from a checker's confident answers (P above 0.9 or below 0.1).

In [ ]:
moved = None
if HAVE["s1_moved"]:
    mv_tr, mv_te = df.query("station == 's1_moved' and pool == 'train'"), df.query("station == 's1_moved' and pool == 'test'")
    ref = s1_test[s1_test.light == "night"]                          # same lamp light, camera not moved
    unsure = lambda f: round(100 * float(f.head_s1_p.between(*UNSURE).mean()), 1)
    pred = {"YOLO": lambda f: f.yolo, "Fast classifier": lambda f: f.head_s1_p > 0.5, "Checker A": lambda f: f.vlm}
    if CHECKER_B: pred["Checker B"] = lambda f: f.idm
    moved = dict(unsure_pct_before=unsure(ref), unsure_pct_after=unsure(mv_te),
                 acc_before_pct={k: round(100 * acc(fn(ref), ref), 1) for k, fn in pred.items()},
                 acc_after_pct={k: round(100 * acc(fn(mv_te), mv_te), 1) for k, fn in pred.items()})
    moved["drift_alarm"] = bool(moved["unsure_pct_after"] > max(2 * moved["unsure_pct_before"], moved["unsure_pct_before"] + 5))
    drop = pd.DataFrame({"station 1, lamp %": moved["acc_before_pct"], "camera moved %": moved["acc_after_pct"]})
    print(drop.assign(drop=drop.iloc[:, 0] - drop.iloc[:, 1]).to_string())
    print(f"\nDrift alarm: the fast classifier was unsure on {moved['unsure_pct_before']}% of frames before the bump and {moved['unsure_pct_after']}% after: "
          f"{'FIRES' if moved['drift_alarm'] else 'quiet'}")
    curves["camera moved"] = {"YOLO retrain": yolo_curve(yolo, yolo_train, mv_tr, mv_te),
                              "Fast classifier, human labels": fc_curve(head_s1, yolo_train, mv_tr, mv_te),
                              **teacher_curves(head_s1, yolo_train, mv_tr, mv_te), **checkers(mv_te)}
    baselines["camera moved"] = always_part(mv_te)
else:
    print("Camera moved: moved_* clips not filmed, so this test is skipped and stored as null.")

## 10. Test 4: new station (cold start)
A second station: a different table, tray and background. It starts with no labels of its own.
1. **Transfer:** the station 1 fast classifier as it is, 0 labels.
2. **Checkers, zero-shot.**
3. **Teacher loop:** the checker labels the new station's training frames, only its confident labels are kept (P above 0.9 or below 0.1), and the fast classifier is fit on those alone. Nobody labels anything. It reports how many labels were confident, how often they were wrong, and **which parts they cover**: confident labels can all be right and still leave out a whole part.
4. **Coverage check on the teacher's labels:** the balance rule from section 2 applied to the checker's answers (the smaller answer needs at least half the count of the larger). In a deployment this is what stops a refit like this one and sends the frames to a person.
5. **Human labels,** for comparison: the same classifier fit on N frames a person labelled.
6. **YOLO** without retraining, then retrained on N frames of the new station.

In [ ]:
teacher = None
if HAVE["s2"]:
    s2_tr, s2_te = df.query("station == 's2' and pool == 'train'"), df.query("station == 's2' and pool == 'test'")
    none = s2_tr.iloc[:0]                                            # a new station starts with no labels of its own
    teacher = {}
    for c, name in CHECKERS:                                         # the teacher loop: every confident checker label, no person
        conf = s2_tr[(s2_tr[f"{c}_p"] > 0.9) | (s2_tr[f"{c}_p"] < 0.1)].assign(teach=lambda f: f[f"{c}_p"] > 0.5)
        said_part, said_empty = int(conf.teach.sum()), int((~conf.teach).sum())
        t = dict(confident_labels=len(conf), of_frames=len(s2_tr),
                 wrong_pct=round(100 * float((conf.teach != conf.part).mean()), 1) if len(conf) else None,
                 said_part=said_part, said_empty=said_empty,
                 coverage_check_pass=bool(min(said_part, said_empty) >= 0.5 * max(said_part, said_empty) > 0),
                 part_labels_by_true_label={k: int(v) for k, v in conf[conf.teach].label.value_counts().items()},   # known here from the clip names, not on site
                 fit_s=None, acc_pct=None)
        if conf.teach.nunique() == 2:
            t0 = time.time(); m = fit(conf, "teach"); t["fit_s"] = round(time.time() - t0, 3)
            t["acc_pct"] = round(100 * acc(p_part(m, s2_te) > 0.5, s2_te), 1)
        teacher[f"checker {name}"] = t
    curves["new layout"] = {"YOLO retrain": yolo_curve(yolo, none, s2_tr, s2_te),
                            "Fast classifier, human labels": fc_curve(head_s1, none, s2_tr, s2_te),
                            **teacher_curves(head_s1, none, s2_tr, s2_te), **checkers(s2_te)}
    baselines["new layout"] = always_part(s2_te)
    m5 = fit(sample(s2_tr, 5, 0))                                    # one of the 5-label refits above (seed 0), kept for the video
    df.loc[df.station == "s2", "s2_after5_p"] = p_part(m5, df[df.station == "s2"])
    L = curves["new layout"]
    print(f"Guessing 'part' scores {baselines['new layout']:.0%} on these test frames.")
    print(f"1. Transfer, the station 1 fast classifier with 0 labels: {L['Fast classifier, human labels'][0]['mean']:.0%}")
    print("2. Checkers, zero-shot: " + ", ".join(f"{k} {v[0]['mean']:.0%}" for k, v in L.items() if k.startswith("Checker") and v))
    for k, t in teacher.items():
        print(f"3. Teacher loop, {k}: {t['confident_labels']} of {t['of_frames']} training frames labelled confidently"
              + (f", {t['wrong_pct']}% of them wrong" if t["confident_labels"] else "")
              + (f". Fast classifier trained on those alone: {t['acc_pct']}%, with no human labels" if t["acc_pct"] is not None else ". Not enough to fit (needs both answers)"))
        if t["confident_labels"]:
            print(f"   It said 'part' {t['said_part']} times and 'empty' {t['said_empty']} times. "
                  f"4. Coverage check: {'PASS' if t['coverage_check_pass'] else 'FAIL: in a deployment this refit is blocked and the frames go to a person'}. "
                  f"(Known here from the clip names: its 'part' labels were {t['part_labels_by_true_label'] or 'none'}.)")
    print("5. Human labels: " + ", ".join(f"N={p['n']}: {p['mean']:.0%} (worst seed {p['min']:.0%})" for p in L["Fast classifier, human labels"][1:]))
    print("6. YOLO: " + ", ".join(f"N={p['n']}: {p['mean']:.0%}" + (f" after {p['s']:.0f} s of training" if p["n"] else " without retraining") for p in L["YOLO retrain"]))
else:
    print("New station: s2_* clips not filmed, so this test is skipped and stored as null.")

## 11. The checkers: blind, or just a badly placed cutoff?
Accuracy at a fixed 0.5 cutoff mixes two things: can the checker tell a loaded tray from an empty one at all, and is its yes/no line in the right place. So two numbers per checker:
1. **Ranking (AUC):** the chance that a random frame with a part gets a higher P(part) than a random empty frame. 50% means blind, 100% means it ranks every frame correctly.
2. **A per-site cutoff:** fit the yes/no line on 20 labelled station 1 training frames (5 draws), then score station 1's test frames with it, using the scoreboard's average. This is what a deployment would do on each site.

In [ ]:
from sklearn.metrics import roc_auc_score

def auc(frames, col):                                                # None when the frames hold only one class
    return round(100 * float(roc_auc_score(frames.part, frames[col])), 1) if frames.part.nunique() == 2 else None

def best_cutoff(p, y):                                               # the cutoff that gets the most of these labelled frames right
    s = np.sort(np.unique(p))
    cands = np.concatenate([[0.0], (s[1:] + s[:-1]) / 2, [1.0]])
    return float(cands[int(np.argmax([((p > c) == y).mean() for c in cands]))])

def cell_mean(pred, frames):                                         # the scoreboard's average: every light x part cell counts the same
    return float(frames.assign(ok=np.asarray(pred) == frames.part.values).groupby(["light", "label"]).ok.mean().mean())

sets = {"station 1 test": s1_test}
if HAVE["s1_moved"]: sets["camera moved test"] = df.query("station == 's1_moved' and pool == 'test'")
if HAVE["s2"]: sets["station 2 test"] = df.query("station == 's2' and pool == 'test'")
s1_labelled = df.query("station == 's1' and pool == 'train'")      # the frames a person would label on site to set the cutoff
checker_eval = {}
for c, name in CHECKERS:
    cuts = [best_cutoff(lab[f"{c}_p"].values, lab.part.values) for lab in (sample(s1_labelled, CUTOFF_N, seed) for seed in SEEDS)]
    e = checker_eval[f"Checker {name}"] = dict(
        auc_pct={k: auc(f, f"{c}_p") for k, f in sets.items()},
        station1_avg_at_05_pct=round(100 * cell_mean(s1_test[f"{c}_p"] > 0.5, s1_test), 1),
        station1_avg_fitted_pct=round(100 * float(np.mean([cell_mean(s1_test[f"{c}_p"] > cut, s1_test) for cut in cuts])), 1),
        station1_avg_fitted_worst_pct=round(100 * float(min(cell_mean(s1_test[f"{c}_p"] > cut, s1_test) for cut in cuts)), 1),
        cutoffs=[round(x, 3) for x in cuts])
    print(f"Checker {name}: ranking (AUC) " + ", ".join(f"{k} {v}%" for k, v in e["auc_pct"].items()))
    print(f"   station 1 average: {e['station1_avg_at_05_pct']}% at the 0.5 cutoff, {e['station1_avg_fitted_pct']}% (worst draw {e['station1_avg_fitted_worst_pct']}%) "
          f"with a cutoff fitted on {CUTOFF_N} labelled frames; cutoffs {e['cutoffs']}")
guess_avg = round(100 * cell_mean(np.ones(len(s1_test), bool), s1_test), 1)
print(f"Guessing 'part' scores {guess_avg}% on the same average.")

## 12. The adaptation table
One row per change and approach: accuracy right after the change, the labels needed to get back to 95% (empty = not reached), and the seconds to adapt. Each change also gets a row for guessing "part". Saved as `adaptation.csv`.
The learning curves show accuracy against labels added, for the new part and the new station. The shaded band runs down to the worst of the 5 seeds.

In [ ]:
TESTS = ["new product", "lighting, night to day", "lighting, day to night", "camera moved", "new layout"]
adapt = []
for test in TESTS:
    for name, pts in curves.get(test, {"all approaches": None}).items():
        hit = next((p for p in pts if p["mean"] >= TARGET), None) if pts else None
        best = max(pts, key=lambda p: p["mean"]) if pts else None
        adapt.append(dict(test=test, approach=name,
                          acc_after_change_pct=round(100 * pts[0]["mean"], 1) if pts else None,
                          labels_to_95=hit["n"] if hit else None,
                          seconds_to_adapt=hit["s"] if hit else None,
                          best_pct=round(100 * best["mean"], 1) if pts else None,
                          note="" if hit else f"not back to 95% (best {100 * best['mean']:.0f}% at N={best['n']})" if pts
                               else "clips not filmed" if test not in curves else "not available, see the printout above"))
    if test in baselines:
        adapt.append(dict(test=test, approach="Always 'part' (guessing)", acc_after_change_pct=round(100 * baselines[test], 1),
                          labels_to_95=None, seconds_to_adapt=None, best_pct=None, note="the score of guessing"))
table = pd.DataFrame(adapt)
table.to_csv(f"{OUT}/adaptation.csv", index=False)
print(table.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, test, title in zip(axes, ("new product", "new layout"), ("new part (whisk)", "new station")):
    ax.set_title(title); ax.set_xlabel("labels added, N"); ax.axhline(100 * TARGET, color="grey", lw=0.8, ls="--")
    for name, pts in curves.get(test, {}).items():
        if pts and len(pts) > 1:
            n, mean, lo = zip(*[(p["n"], 100 * p["mean"], 100 * p["min"]) for p in pts])
            ax.plot(n, mean, marker="o", label=name); ax.fill_between(n, lo, mean, alpha=0.15)
    if test in baselines: ax.axhline(100 * baselines[test], color="black", lw=0.8, ls=":", label="always 'part' (guessing)")
    if test in curves: ax.legend(fontsize=8)
    else: ax.text(0.5, 0.5, "clips not filmed", ha="center", transform=ax.transAxes)
axes[0].set_ylabel("accuracy on that condition's test frames, %")
plt.tight_layout(); plt.savefig(f"{OUT}/learning_curve.png", dpi=150); plt.show()

## 13. Speed on this GPU, and the Jetson AGX Orin 32GB estimate
Timings use batches of 10 frames (one per camera). "Writes a 60-token JSON" asks checker A for a 6-field JSON answer, which is what a naive VLM integration would do. "6 questions one at a time" is **not timed**: it is 6 times the 1-question time. Checker B is timed with 1 question and with 6 questions in one request, **one frame at a time**, because its loader takes one frame per call; so the A vs B speed comparison is not like for like.
**The gated load:** the checker only sees frames where the fast classifier is unsure or its answer just changed, plus a 3% random audit of the rest. That share is measured on the station 1 test frames, and each gated row is that share times the checker's cost per frame.

In [ ]:
def timed(fn, n=3):
    fn(); torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / n

ten = list(s1_test.path.iloc[:10])
JSON_Q = "Describe the tray as JSON with keys part_on_tray, hand_visible, tray_visible, lighting, object_type, confidence."
@torch.no_grad()
def talk():
    vlm.generate(**vlm_inputs(ten, JSON_Q), max_new_tokens=60, min_new_tokens=60, do_sample=False)

ms = {
    "YOLO classifier": timed(lambda: list(yolo.predict(ten, imgsz=224, verbose=False))) * 100,
    "Fast classifier": timed(lambda: head.predict_proba(embed(ten))) * 100,
    "Checker A decides, 1 question": timed(lambda: p_yes(ten)) * 100,
    "Checker A writes a 60-token JSON": timed(talk, n=2) * 100,
}
ms["Checker A, 6 questions one at a time (6 x 1 question)"] = 6 * ms["Checker A decides, 1 question"]
if CHECKER_B:
    ms["Checker B decides, 1 question"] = timed(lambda: [ask(p, Q1) for p in ten]) * 100
    ms["Checker B decides, 6 questions in one request"] = timed(lambda: [ask(p, Q6) for p in ten]) * 100
timing_notes = {"Checker A, 6 questions one at a time (6 x 1 question)": "not timed: 6 x the 1-question time",
                "Checker B decides, 1 question": "timed one frame at a time (checker A in batches of 10)",
                "Checker B decides, 6 questions in one request": "timed one frame at a time (checker A in batches of 10)"}

# the gate on station 1 test frames: fast classifier unsure, or its answer just changed, plus a 3% audit of the rest
state = (s1_test.head_s1_p > 0.5).astype(int)
flagged = float((s1_test.head_s1_p.between(*UNSURE) | state.groupby(s1_test["clip"]).diff().fillna(0).ne(0)).mean())
gate_share_pct = round(100 * (flagged + AUDIT * (1 - flagged)), 1)
for k in ("Checker A decides, 1 question", "Checker B decides, 6 questions in one request"):
    if k in ms: ms[f"{k}, gated ({gate_share_pct}% of frames)"] = gate_share_pct / 100 * ms[k]

need = CAMERAS * CAMERA_FPS
print(f"GPU: {torch.cuda.get_device_name()} | need {need:g} frames/s | Orin estimate = T4 x {ORIN_FACTOR} | gate sends {gate_share_pct}% of frames to the checker\n")
budget = []
for k, v in ms.items():
    orin = v * ORIN_FACTOR
    budget.append(dict(approach=k, t4_ms=round(v, 1), orin_ms_est=round(orin, 1), orin_frames_per_s=round(1000 / orin, 1) if orin else None, orin_gpu_share_pct=round(need * orin / 10)))
print(pd.DataFrame(budget).to_string(index=False))
fast = next(b for b in budget if b["approach"] == "Fast classifier")
for b in budget:
    if "gated" in b["approach"]: print(f"Fast classifier on every frame + {b['approach']}: about {fast['orin_gpu_share_pct'] + b['orin_gpu_share_pct']}% of the Orin GPU")
print("\nGPU share over 100% = does not fit. Orin figures are estimates from a T4, not measurements.")

## 14. Save the results
Downloads a zip with `results.json`, `adaptation.csv`, `predictions.csv` (every frame, every model, for the video notebook), `cycle.csv`, `learning_curve.png`, `samples.png`, `cycle.png` and one frame per clip. DEMO numbers come from synthetic clips and mean nothing; the case study tooling refuses them.

In [ ]:
import transformers
samp = df[df.pool == "test"].groupby("clip", sort=False).head(1)
nrow = -(-len(samp) // 3)
fig, axes = plt.subplots(nrow, 3, figsize=(13, 2.9 * nrow), squeeze=False)
for a in axes.flat: a.axis("off")
for a, r in zip(axes.flat, samp.itertuples()):
    a.imshow(Image.open(r.path))
    fp = r.head_p if r.station == "s1" else r.head_s1_p
    a.set_title(f"{r.clip}\nYOLO: {'part' if r.yolo else 'empty'} | fast P(part) {fp:.2f} | A P(part) {r.vlm_p:.2f}" + (f" | B P(part) {r.idm_p:.2f}" if CHECKER_B else ""), fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/samples.png", dpi=150); plt.show()

os.makedirs(f"{OUT}/frames", exist_ok=True)
for r in samp.itertuples(): shutil.copy(r.path, f"{OUT}/frames/{r.clip}.jpg")
cols = [c for c in ("clip", "station", "light", "label", "pool", "part", "yolo", "yolo_p", "head_p", "head_s1_p", "vlm_p", "idm_p", "s2_after5_p") if c in df]
(df.assign(frame=df.path.map(os.path.basename), t_s=df.path.map(lambda p: (int(os.path.basename(p)[:4]) - 1) / FPS))[["frame", "t_s"] + cols]
   .round(4).to_csv(f"{OUT}/predictions.csv", index=False))
json.dump(dict(
    demo=DEMO, gpu=torch.cuda.get_device_name(), vlm=VLM_ID, checker_b=IDM_ID, checker_b_available=CHECKER_B, encoder=ENC_ID,
    question=QUESTION, fps_extracted=FPS, skip_s=SKIP_S, run_utc=time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
    versions=dict(torch=torch.__version__, transformers=getattr(transformers, "__version__", "?")),
    frames=df.groupby(["station", "light", "label", "pool"]).size().rename("n").reset_index().to_dict("records"),
    accuracy_pct=(score * 100).round(1).reset_index().astype(str).to_dict("records"),
    ship_gate=gate, yolo_retrain_s=round(yolo_train_s), head_fit_s=round(head_fit_s, 2), fast_c=FAST_C, embed_ms_per_frame=round(embed_ms, 1),
    vlm_tokens_per_frame=TOKENS_PER_FRAME, budget=budget, timing_notes=timing_notes, orin_factor=ORIN_FACTOR,
    cycle_loads_counted=loads, cycle_load_times_s=load_times,
    adaptation=adapt, learning_curves=curves, baselines_pct={k: round(100 * v, 1) for k, v in baselines.items()},
    gate_share_pct=gate_share_pct, camera_moved=moved, teacher_loop=teacher, checker_eval=checker_eval, guess_avg_pct=guess_avg,
), open(f"{OUT}/results.json", "w"), indent=2, default=str)
shutil.make_archive(f"edge_vision_{OUT}", "zip", OUT)
try:
    from google.colab import files; files.download(f"edge_vision_{OUT}.zip")
except ImportError:
    print("saved", f"edge_vision_{OUT}.zip")